In [1]:
# Importação das bibliotecas
from scipy.stats import loguniform, uniform
import numpy as np
import pandas as pd
import os

from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import VGG16
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.utils import shuffle
from sklearn.model_selection import RandomizedSearchCV

from SwarmPackagePy import gwo
import pyswarms as ps

In [2]:
# Tratamento das imagens
def create_street_data(path, street_types, im_size):
    images, labels = [], []
    streets = [(item, os.path.join(path, item, street)) 
               for item in street_types 
               for street in os.listdir(os.path.join(path, item))]
    streets_df = pd.DataFrame(streets, columns=['street type', 'image'])
    
    for _, row in streets_df.iterrows():
        img = load_img(row['image'], target_size=(im_size, im_size))
        images.append(img_to_array(img))
        labels.append(row['street type'])
    
    return np.array(images, dtype='float32') / 255.0, np.array(labels)

In [3]:
im_size = 224

street_types = ['clean', 'litter', 'recycle']
path = '../Dataset_Menor/'
path_test = '../Dataset_Test/'

In [ ]:
train_images, train_labels = create_street_data(path, street_types, im_size)
test_images, test_labels = create_street_data(path_test, street_types, im_size)

streets_count = pd.value_counts(train_labels)
print("Streets in each category:", streets_count)

In [ ]:
label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform(train_labels)
test_labels_encoded = label_encoder.transform(test_labels)

train_x, val_x, train_y, val_y = train_test_split(train_images, train_labels_encoded, test_size=0.15, random_state=415)

print(f"Train shape: {train_x.shape}")
print(f"Validation shape: {val_x.shape}")
print(f"Test shape: {test_images.shape}")

In [6]:
# Definição, compilação e treinamento
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(im_size, im_size, 3))

for layer in base_model.layers:
    layer.trainable = False

In [7]:
# Implementação da Grid Search
def create_model(learning_rate, dropout_rate):
    top_model = Sequential([
        Flatten(input_shape=base_model.output_shape[1:]),
        Dropout(dropout_rate),
        Dense(3, activation='softmax')
    ])
    
    model = Model(inputs=base_model.input, outputs=top_model(base_model.output))
    
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [8]:
model = KerasClassifier(build_fn=create_model, epochs=5, verbose=0)

param_grid = {
    'learning_rate': [0.00001, 0.0001, 0.001, 0.01, 0.1],
    'dropout_rate': [0.0, 0.1, 0.25, 0.4, 0.5]
}

In [ ]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1, cv=3)

grid_result = grid.fit(train_x, train_y)

results = []

for params, mean_test_score in zip(grid_result.cv_results_['params'], grid_result.cv_results_['mean_test_score']):
    results.append({'Hiperparâmetros': params, 'Perda': 1 - mean_test_score})

best_learning_rate = grid_result.best_params_['learning_rate']
best_dropout_rate = grid_result.best_params_['dropout_rate']
print("Melhor: %f usando %s" % (grid_result.best_score_, grid_result.best_params_))

In [ ]:
# Definição, compilação e treinamento
base_model_output = base_model.output
top_layers_output = Flatten()(base_model_output)
top_layers_output = Dropout(best_dropout_rate)(top_layers_output)
top_layers_output = Dense(3, activation='softmax')(top_layers_output)

final_model = Model(inputs=base_model.input, outputs=top_layers_output)

final_model.compile(optimizer=Adam(best_learning_rate),
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

history = final_model.fit(train_x, 
                          train_y, 
                          epochs=5, 
                          validation_data=(val_x, val_y))

In [ ]:
# Avaliação e análise
best_results = []
best_results.append({'Melhor taxa de aprendizado': best_learning_rate, 'Melhor taxa de dropout': best_dropout_rate})

print(f"Melhor taxa de aprendizado: {best_learning_rate}")
print(f"Melhor taxa de dropout: {best_dropout_rate}")

test_loss, test_accuracy = final_model.evaluate(test_images, test_labels_encoded, verbose=1)
print(f"\nAcurácia no teste: {test_accuracy*100:.2f}%\n")

y_pred_test = final_model.predict(test_images)
y_pred_test_classes = np.argmax(y_pred_test, axis=1)

confusion_mtx = confusion_matrix(test_labels_encoded, y_pred_test_classes)
print("Matriz de Confusão:")
print(confusion_mtx)

class_report = classification_report(test_labels_encoded, y_pred_test_classes, target_names=label_encoder.classes_)
print("\nRelatório de Classificação:")
print(class_report)

y_true_binarized = to_categorical(test_labels_encoded)

for i in range(y_true_binarized.shape[1]):
    auc_score = roc_auc_score(y_true_binarized[:, i], y_pred_test[:, i])
    print(f'AUC para a classe {label_encoder.classes_[i]}: {auc_score:.2f}')

In [ ]:
# Armazenamento do Excel e Modelo
shapes_data = {
    'Conjunto': ['Treino', 'Validação', 'Teste'],
    'Formato': [train_x.shape, val_x.shape, test_images.shape]
}
df_shapes = pd.DataFrame(shapes_data)

df_results = pd.DataFrame(results)

df_best_results = pd.DataFrame(best_results)

df_confusion_mtx = pd.DataFrame(confusion_mtx, index=label_encoder.classes_, columns=label_encoder.classes_)

report_data = classification_report(test_labels_encoded, y_pred_test_classes, target_names=label_encoder.classes_, output_dict=True)
df_class_report = pd.DataFrame(report_data).transpose()

auc_scores = {label_encoder.classes_[i]: roc_auc_score(y_true_binarized[:, i], y_pred_test[:, i]) for i in range(y_true_binarized.shape[1])}
df_auc_scores = pd.DataFrame(list(auc_scores.items()), columns=['Classe', 'AUC'])

with pd.ExcelWriter('Grid_Excel.xlsx') as writer:
    df_shapes.to_excel(writer, sheet_name='Formas dos Conjuntos', index=False)
    df_results.to_excel(writer, sheet_name='Resultados Otimizacao', index=False)
    df_best_results.to_excel(writer, sheet_name='Melhor Resultado', index=False)
    df_confusion_mtx.to_excel(writer, sheet_name='Matriz de Confusão')
    df_class_report.to_excel(writer, sheet_name='Relatório de Classificação')
    df_auc_scores.to_excel(writer, sheet_name='AUC por Classe', index=False)

print("Resultados exportados para o arquivo 'Grid_Excel.xlsx'")

In [ ]:
model_save_path = 'Grid_SaveModel.h5'

final_model.save(model_save_path)
print(f"Modelo salvo com sucesso em: {model_save_path}")